In [1]:
import os
import shutil
import datetime
import math
import re
from functools import partial
from itertools import product
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial import KDTree, cKDTree, distance_matrix
from scipy.integrate import trapz
from scipy import stats, spatial

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

import tensorflow as tf

import joblib
import psutil

from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing as mp

# custom modules
import zsdm_utils_v2 as zsdm_utils

# Number of physical CPU cores
n_phys_cores = psutil.cpu_count(logical=False)

In [2]:
data = pd.read_csv('TiZrNbON.csv')
#print(data)
rrange_file = 'TiZrNbON.RRNG'
ions, rrngs = zsdm_utils.read_rrng(rrange_file)
data = data.rename(columns={'Mass': 'Da', 'Position_0': 'x','Position_1': 'y','Position_2': 'z'})
data = data.drop(columns=['Unnamed: 4'])
data = data[['x', 'y', 'z','Da']]
columns_to_round = ['x', 'y', 'z']
data[columns_to_round] = data[columns_to_round].round(5)
def atom_filter(x, Atom_range):
    Atom_total = pd.DataFrame()
    for i in range(len(Atom_range)):
        Atom = x[x['Da'].between(Atom_range['lower'][i], Atom_range['upper'][i], inclusive='both')]
        Atom_total = pd.concat([Atom_total, Atom])
        # Count_Atom= len(Atom_total['Da'])   
    return Atom_total[['x','y','z','Da']]

In [3]:
#Pls type me
element_1_name = 'Ti'
element_2_name = 'Zr'
element_3_name = 'Nb'
element_4_name = 'Zr O'
element_5_name = 'O'
element_1_range = rrngs[rrngs['comp']==element_1_name+':1']
element_2_range = rrngs[rrngs['comp']==element_2_name+':1']
element_3_range = rrngs[rrngs['comp']==element_3_name+':1']
element_4_range = rrngs[rrngs['comp'] == 'Zr:1 O:1']
element_5_range = rrngs[rrngs['comp']==element_5_name+':1']
element_6_range = rrngs[rrngs['comp']=='N:1']
element_7_range = rrngs[rrngs['comp']=='Zr:1 N:1']
#print(element_1_range)
#print(element_2_range)
element_1 = atom_filter(data, element_1_range)
element_2 = atom_filter(data, element_2_range)
element_3 = atom_filter(data, element_3_range)
element_O_Zr= atom_filter(data, element_4_range)
element_O = atom_filter(data, element_5_range)
element_6 = atom_filter(data, element_6_range)
element_7 = atom_filter(data, element_7_range)

C:\Users\Yinjiawei\AppData\Local\Temp\ipykernel_8336\3746915449.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Atom = x[x['Da'].between(Atom_range['lower'][i], Atom_range['upper'][i], inclusive='both')]
C:\Users\Yinjiawei\AppData\Local\Temp\ipykernel_8336\3746915449.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Atom = x[x['Da'].between(Atom_range['lower'][i], Atom_range['upper'][i], inclusive='both')]
C:\Users\Yinjiawei\AppData\Local\Temp\ipykernel_8336\3746915449.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as

In [4]:
element_1["source"] = 0
element_2["source"] = 1
element_3["source"] = 2
element_N = pd.concat([element_6, element_7], ignore_index=True)   # Merge element_6 and element_7
element_O = pd.concat([element_O_Zr, element_O], ignore_index=True) # Merge element_O_Zr and element_O
element_O["source"] = 3
element_N["source"] = 4
# Concatenate all DataFrames
combined_df = pd.concat([element_1, element_2, element_3, element_O, element_N], ignore_index=True)

In [5]:
# Define translation amounts
dx, dy, dz = 17.5, 19.05, 24.22

# Translate only the x, y, z columns
combined_df[['x', 'y', 'z']] += np.array([dx, dy, dz])
columns_to_round = ['x', 'y', 'z']
combined_df[columns_to_round] = combined_df[columns_to_round].round(5)

# Define filtering criteria
mask = (
    (combined_df['x'] >= 0) & (combined_df['x'] <= 25) &
    (combined_df['y'] >= 0) & (combined_df['y'] <= 25) &
    (combined_df['z'] >= 0) & (combined_df['z'] <= 25)
)

# Filter to obtain data within the cube region
cube_data_total = combined_df[mask]
cube_data = cube_data_total[['x', 'y', 'z', 'source']]
data = cube_data.values

In [6]:
cube_data = cube_data_total[['x', 'y', 'z', 'source']]
data = cube_data.values

In [7]:

# Given parameters
N_1NN = 8           # BCC structure: 12 first nearest neighbors
eta = 0.5            # detection efficiency
lambda_1 = N_1NN * eta  # expected number of detected 1NNs

# Maximum m to calculate (APT mNN up to m = 20)
max_m = 20

# Compute P(k; lambda) for k = 0 to (max_m)
P_k = [math.exp(-lambda_1) * (lambda_1 ** k) / math.factorial(k) for k in range(max_m + 1)]

# Compute cumulative probabilities P_>=m
P_ge_m = []
for m in range(1, max_m + 1):
    cumulative = sum(P_k[:m])
    P_ge = 1 - cumulative
    P_ge_m.append(P_ge)

# Build the table
df = pd.DataFrame({
    "APT mNN (m)": list(range(1, max_m + 1)),
    "P(≥m; λ=6)": P_ge_m
})
df.head(20)  # Show the full 20-row table

,APT mNN (m),P(≥m; λ=6)
0,1,9.816844e-01
1,2,9.084218e-01
2,3,7.618967e-01
3,4,5.665299e-01
4,5,3.711631e-01
5,6,2.148696e-01
6,7,1.106740e-01
7,8,5.113362e-02
8,9,2.136343e-02
9,10,8.132243e-03


# Construct voxel grid center points

In [8]:
data_min = cube_data.min()
data_max = cube_data.max()

#print(data_min)
#print(data_max)
data_min['z'], data_min['y'], data_min['x'] = 0, 0, 0 #nm
data_max['z'], data_max['y'], data_max['x'] = 25, 25, 25   #nm

# scanning parameters 
voxel = np.array([1.0],dtype=np.float64)
#print("voxel shape {}".format(voxel.shape))
stride = 0.5

In [9]:
data_Z_list = list(np.arange(int(data_min['z']), int(data_max['z']), stride))
data_Y_list = list(np.arange(int(data_min['y']), int(data_max['y']), stride))
data_X_list = list(np.arange(int(data_min['x']), int(data_max['x']), stride))
data_sphere_points = np.zeros((1,3))
for data_Z, data_Y, data_X in product(data_Z_list, data_Y_list, data_X_list):
    if data_Z+voxel > data_max['z'] or data_Y+voxel  > data_max['y'] or data_X+voxel  > data_max['x']:
        continue
    else:
        temp = np.array([data_X+voxel/2, data_Y+voxel/2, data_Z+voxel/2]).reshape((1,3)) 
        data_sphere_points = np.concatenate((data_sphere_points, temp), axis=0)
data_sphere_points = data_sphere_points[1:]

In [10]:
# Example parameter settings
element_fractions = {0: 0.56, 1: 0.30, 2: 0.14}
target_pairs = [(0, 1), (1, 0), (2, 2)]  # Ti-Zr, Zr-Ti, Nb-Nb
structure_type_global = "SRO"
r_core = 0.75

In [11]:
# === Vectorized in_voxel_mask ===
def in_voxel_mask(positions, voxel_bounds):
    return (
        (voxel_bounds[0][0] <= positions[:, 0]) & (positions[:, 0] <= voxel_bounds[0][1]) &
        (voxel_bounds[1][0] <= positions[:, 1]) & (positions[:, 1] <= voxel_bounds[1][1]) &
        (voxel_bounds[2][0] <= positions[:, 2]) & (positions[:, 2] <= voxel_bounds[2][1])
    )

# === Single voxel computation ===
def process_voxel(voxel_center, data, tree, neighbors_idx, P_ge_m,
                  r_core, element_fractions, target_pairs, max_m_used):
    voxel_bounds = (
        (voxel_center[0] - 0.5, voxel_center[0] + 0.5),
        (voxel_center[1] - 0.5, voxel_center[1] + 0.5),
        (voxel_center[2] - 0.5, voxel_center[2] + 0.5)
    )

    indices_in_voxel = tree.query_ball_point(voxel_center, r=np.sqrt(3))
    sro_list = []

    for i in indices_in_voxel:
        pos = data[i, :3]
        if np.linalg.norm(pos - voxel_center) > r_core:
            continue

        center_type = int(data[i, 3])
        neighbor_ids = neighbors_idx[i, 1:max_m_used+1]

        mask = in_voxel_mask(data[neighbor_ids, :3], voxel_bounds)
        neighbor_types = data[neighbor_ids[mask], 3].astype(int)
        applied_weights = P_ge_m[mask]

        if applied_weights.size == 0:
            continue

        weighted_counts = {}
        for t, w in zip(neighbor_types, applied_weights):
            pair_key = (center_type, t)
            if pair_key in target_pairs:
                weighted_counts[pair_key] = weighted_counts.get(pair_key, 0) + w

        total_weight = applied_weights.sum()
        sro_values = {}
        for (a, b) in target_pairs:
            if center_type != a:
                continue
            P_AB = weighted_counts.get((a, b), 0.0) / total_weight if total_weight > 0 else np.nan
            x_B = element_fractions.get(b, 1e-8)
            alpha_AB = P_AB / x_B if x_B > 0 else np.nan
            sro_values[f"{a}-{b}"] = alpha_AB

        sro_list.append(sro_values)

    # === Aggregation ===
    feature_vector = []
    for (a, b) in target_pairs:
        vals = [atom.get(f"{a}-{b}", np.nan) for atom in sro_list]
        mean_val = np.nanmean(vals) if len(vals) > 0 else 0.0
        feature_vector.append(mean_val)

    return feature_vector


# === Main function (parallelized) ===
def compute_sro_features_parallel(data, data_sphere_points, P_ge_m,
                                  r_core, element_fractions, target_pairs, n_jobs=-1):
    tree = cKDTree(data[:, :3])
    max_m_used = sum(p >= 0.01 for p in P_ge_m)
    neighbors_dist, neighbors_idx = tree.query(data[:, :3], k=max_m_used+1)

    P_ge_m = np.array(P_ge_m[:max_m_used])  # Trim invalid weights

    # ⚡ Use threading backend + batch_size to avoid workload imbalance
    results = Parallel(n_jobs=n_jobs, backend="threading", batch_size=20)(
        delayed(process_voxel)(
            voxel_center, data, tree, neighbors_idx, P_ge_m,
            r_core, element_fractions, target_pairs, max_m_used
        )
        for voxel_center in tqdm(data_sphere_points, desc="Processing voxels", unit="voxel")
    )
    return np.array(results)

In [12]:
X_total = compute_sro_features_parallel(
    data, data_sphere_points, P_ge_m,
    r_core=0.75,
    element_fractions=element_fractions,
    target_pairs=target_pairs,
    n_jobs=-1   # Number of parallel cores, -1 means use all available
)

Processing voxels:   1%|▋                                                    | 1640/117649 [00:06<07:38, 253.28voxel/s]C:\Users\Yinjiawei\AppData\Local\Temp\ipykernel_8336\3881508743.py:58: RuntimeWarning: Mean of empty slice
  mean_val = np.nanmean(vals) if len(vals) > 0 else 0.0
Processing voxels: 100%|███████████████████████████████████████████████████| 117649/117649 [10:40<00:00, 183.73voxel/s]


In [ ]:
# 1. Load the saved scaler and model
scaler = joblib.load('scaler_Nb.pkl')
rf_model = joblib.load('rf_model_Nb.pkl')

# 2. Load the experimental data to be predicted (N×3 features)
X_exp = X_total 

# 3. Standardize using the scaler fitted on training data
X_exp_std = scaler.transform(X_exp)

# 4. Predict
y_pred = rf_model.predict(X_exp_std)
y_proba = rf_model.predict_proba(X_exp_std)[:, 1]

# 5. Save prediction results

In [ ]:
data_min = cube_data.min()
data_max = cube_data.max()

#print(data_min)
#print(data_max)
data_min['z'], data_min['y'], data_min['x'] = 0, 0, 0 #nm
data_max['z'], data_max['y'], data_max['x'] = 25, 25, 25   #nm
voxel = 1   #nm   delta
stride = 0.5   #nm   delta_voxels
count = 0*8
threshold = 3.75
ite = 0.25


In [ ]:
def SRO_extract_only_lower(lower):
    print('Threshold is above ', lower)
    dspp_filter = dspp.loc[(dspp['d'] > lower) , ['a','b','c','d']]  
    tree = []
    tree = spatial.KDTree(cube_data_total[['x','y','z']])
    index_voxel_sphere = tree.query_ball_point(dspp_filter[['a','b','c']], stride/2*1.414)     
    myList = range(0,len(index_voxel_sphere))
    pre_extract_v1 = pd.DataFrame(np.arange(4).reshape(1,4),columns=['a','b','c','d'])
    for i in myList:
        df_2 = cube_data_total.iloc[index_voxel_sphere[i,]]
        pre_extract_v1 = pd.concat([pre_extract_v1, df_2], ignore_index=True)
    pre_extract_v1 = pre_extract_v1[1:].drop_duplicates().values
    return pre_extract_v1



In [ ]:
y_proba = y_proba.reshape(-1,1)
data_predictions = y_proba
# row_number = data.shape[0]
# detect_eff = 0.52
n_dim = data_predictions.shape[1] #0 matrix 1 CSRO
#%% reshape data_predictions
img_dim_a = int((data_max['x']-data_min['x'])/1)
img_dim_b = int((data_max['y']-data_min['y'])/1)
img_dim_c = int((data_max['z']-data_min['z'])/1)   
conv_shape_a = int((img_dim_a-voxel)/stride)+1
conv_shape_b = int((img_dim_b-voxel)/stride)+1
conv_shape_c = int((img_dim_c-voxel)/stride)+1

conv_probability = np.zeros([conv_shape_c, conv_shape_b, conv_shape_a])   

# for sro_type in dict:   
#     print(dict[sro_type])
conv_probability[:,:,:] = np.reshape(data_predictions[:, 0], (conv_shape_c, conv_shape_b, conv_shape_a))
# print(conv_probability[:,:,:,len1])
#%% calculate the probability of each voxel (stride*stride*4) 
starttime = datetime.datetime.now()

pixel_probability = np.zeros([int(img_dim_c/stride), int(img_dim_b/stride), int(img_dim_a/stride)])    #1
pixel_z = 0
pixel_y = 0
pixel_x = 0
count = 0
intervial= int(voxel/stride)
# print ('count', count)
# print('pixel_x=', pixel_x)
# print('pixel_y=', pixel_y)
# print('pixel_z=', pixel_z)
# for pixel_dim in range (n_dim):
#     print ('Dimension=', pixel_dim)
for data_Z in np.arange(int(data_min['z']), int(data_max['z']), stride):
    if data_Z+stride > data_max['z']:
        continue
    else:
        for data_Y in np.arange(int(data_min['y']),int(data_max['y']), stride):
            if data_Y+stride > data_max['y']:
                continue
            else:
                for data_X in np.arange(int(data_min['x']), int(data_max['x']), stride):
                    if data_X+stride > data_max['x']:
                        continue
                    else:

                        for k in range (pixel_z, pixel_z-intervial, -1):
                            if k < 0 or k >= conv_shape_c: 
                                continue
                            else:
                                for j in range (pixel_y, pixel_y-intervial, -1):
                                    if j < 0 or j >= conv_shape_b:
                                        continue
                                    else:                                       
                                        for i in range (pixel_x, pixel_x-intervial, -1):
                                            if i < 0 or i >= conv_shape_a:
                                                continue
                                            else:

                                                temp = conv_probability[k, j, i]
                                                pixel_probability[pixel_z, pixel_y, pixel_x] = pixel_probability[pixel_z, pixel_y, pixel_x] + temp
#                                                     print ('i=',i, "j=", j, "k=", k)
                                                count = count + 1
                        pixel_probability[pixel_z, pixel_y, pixel_x] = pixel_probability[pixel_z, pixel_y, pixel_x]
#                             print ('count', count)
                        count=0        
                        # print ("------>>")
                        if pixel_x == int(img_dim_a/stride-1):
                            pixel_x = 0  
                        else:
                            pixel_x = pixel_x + 1
#                             print('pixel_x=', pixel_x)
#                             print('pixel_y=', pixel_y)
#                             print('pixel_z=', pixel_z)
                if pixel_y == int(img_dim_b/stride-1):
                    pixel_y = 0                        
                else:
                    pixel_y = pixel_y + 1
#                     print('pixel_x=', pixel_x)
#                     print('pixel_y=', pixel_y)
#                     print('pixel_z=', pixel_z)
        if pixel_z == int(img_dim_c/stride-1):
            pixel_z = 0                        
        else:
            pixel_z = pixel_z + 1
#             print('pixel_x=', pixel_x)
#             print('pixel_y=', pixel_y)
#             print('pixel_z=', pixel_z)
endtime = datetime.datetime.now()
print ('1st running time is', endtime-starttime)
#%% Corresponding the probability of each small voxel with each data point, and then saved into .txt file
starttime = datetime.datetime.now()

dim_a = int((data_max['x']-data_min['x'])/stride)
dim_b = int((data_max['y']-data_min['y'])/stride)
dim_c = int((data_max['z']-data_min['z'])/stride)
pixel_probability_flatten=np.reshape(pixel_probability, (dim_a*dim_b*dim_c, 1))
# np.save('pixel_probability_flatten_'+element_name_chosen+element_name_chosen, pixel_probability_flatten)

try:
    data_sphere_points = np.load('data_sphere_points_cocrni_%s_%s.npy'%(int(data_min['c']), int(data_max['c'])))
except:
    data_Z_list = list(np.arange(int(data_min['z']), int(data_max['z']), stride))
    data_Y_list = list(np.arange(int(data_min['y']), int(data_max['y']), stride))
    data_X_list = list(np.arange(int(data_min['x']), int(data_max['x']), stride))

    data_sphere_points = np.zeros((1,3))
    for data_Z, data_Y, data_X in product(data_Z_list, data_Y_list, data_X_list):
        if data_Z+stride > data_max['z'] or data_Y+stride > data_max['y'] or data_X+stride > data_max['x']:
            continue
        else:
            temp = np.array([data_X+stride/2, data_Y+stride/2, data_Z+stride/2]).reshape((1,3)) 
            data_sphere_points = np.concatenate((data_sphere_points, temp), axis=0)
    data_sphere_points = data_sphere_points[1:]
#         np.save ('data_sphere_points_FeMnCoCrSi_%s_%s.npy'%(int(data_min['c']), int(data_max['c'])), data_sphere_points)

data_sphere_points_probability = np.concatenate((data_sphere_points, pixel_probability_flatten), axis=1)
dspp = pd.DataFrame(data_sphere_points_probability, columns=['a','b','c','d'])
# dspp.to_excel(f'{element_name_chosen}.xlsx')
#output using different thresholds
# pre_extract = Parallel(n_jobs=-1, verbose=1)(delayed(SRO_extract)(i) for i in myList)
data_SRO = SRO_extract_only_lower(3.1)

In [ ]:
df_np = data_SRO[:,4:]
#%%Calculate CSRO data from DBSCAN
df_np_preprocess = np.concatenate((df_np[:,0].reshape(-1, 1), df_np[:,1].reshape(-1, 1), df_np[:,2].reshape(-1, 1)), axis=1) #xyz is nm scale and do not standard scale.
db = DBSCAN(eps=0.4, min_samples=3).fit(df_np_preprocess)
labels = db.labels_

labels_del_1 = labels[labels!=-1] #Noisy samples are given the label -1
counts = Counter(labels_del_1)
atom_number_each_SRO =  np.fromiter(counts.values(), dtype=int)

df_with_labels = np.concatenate((df_np, labels.reshape(-1, 1)), axis=1)
df_with_labels_filter= df_with_labels[(df_with_labels[:,4]!=-1)]  
# np.savetxt(sro_type + 
#             '_DBSCAN_%s_%s_10_10_above_%s_win_%s_stride_%s.csv'
plot_3d = True
#             , df_with_labels_filter[:, [0, 1, 2, 4]], delimiter=',', fmt='%.3f')


In [ ]:
df = pd.DataFrame(df_with_labels_filter,columns = ["x", "y", "z", 'mass', "type", "cluster_id"])